In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Load data using a relative path
df_q3 = pd.read_csv('../data/q3_retail_promotions.csv')

# Convert to datetime
df_q3['transaction_date'] = pd.to_datetime(df_q3['transaction_date'])

# Extract features
df_q3['year'] = df_q3['transaction_date'].dt.year
df_q3['month'] = df_q3['transaction_date'].dt.month
df_q3['day_of_week'] = df_q3['transaction_date'].dt.dayofweek

# Binary feature: Month end (1 if day >= 25, else 0)
df_q3['is_month_end'] = (df_q3['transaction_date'].dt.day >= 25).astype(int)

# Display sample
print("Date features extracted:")
display(df_q3[['transaction_date', 'year', 'month', 'day_of_week', 'is_month_end']].head())

In [ ]:
# Sort data by date first
df_q3 = df_q3.sort_values('transaction_date')

# 80% Train, 20% Test split
split_idx = int(len(df_q3) * 0.8)
train_df = df_q3.iloc[:split_idx]
test_df = df_q3.iloc[split_idx:]

# Define features and target
X_train = train_df.drop(['items_sold', 'transaction_date'], axis=1)
y_train = train_df['items_sold']
X_test = test_df.drop(['items_sold', 'transaction_date'], axis=1)
y_test = test_df['items_sold']

print(f"Training records: {len(X_train)}")
print(f"Testing records: {len(X_test)}")

In [ ]:
# Identify columns
cat_cols = ['promotion_type', 'location_type', 'store_size']
num_cols = [c for c in X_train.columns if c not in cat_cols]

# Define ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols)
    ])

print("Preprocessing pipeline defined.")

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42)
}

for name, model in models.items():
    # Build the full pipeline
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', model)])
    
    # Fit model
    pipe.fit(X_train, y_train)
    
    # Predict
    y_pred = pipe.predict(X_test)
    
    # Metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    
    print(f"\n--- {name} Results ---")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE: {mae:.2f}")
    
    # Parity Plot
    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, y_pred, alpha=0.5, color='blue')
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    plt.xlabel('Actual Items Sold')
    plt.ylabel('Predicted Items Sold')
    plt.title(f'Parity Plot: {name}')
    plt.show()

    # Feature Importance (For Random Forest)
    if name == "Random Forest":
        importances = pipe.named_steps['regressor'].feature_importances_
        cat_features = pipe.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(cat_cols)
        all_features = list(cat_features) + num_cols
        feat_imp = pd.Series(importances, index=all_features).sort_values(ascending=False)
        
        print("\nTop 5 Most Influential Features:")
        print(feat_imp.head(5))